# CS-323 Artificial Intelligence — Complex Engineering Problem
## Intelligent Vacuum-Cleaning Robot

**Group members**

| Roll No. | Name |
|----------|------|
| CS24105 | Sabika Abbas |
| CS24110 | Alishba Javed |
| CS24114 | Bushra Khalid |
| CS24012 | Maryam Siddiqa |

---

**Program flow:** `Schedule.txt` → **CSP** (cleaning order) → `Environment.txt` → **Graph-based search** (navigation) → **Solution**

**What this notebook does**
1. Loads the grid environment from `Environment.txt` and the scheduling constraints from `Schedule.txt`. Nothing is hardcoded.
2. Uses a **CSP** (backtracking with MRV and the degree heuristic, LCV ordering and forward checking) to assign every dirty location to a time slot.
3. Uses **BFS** (uninformed) and **A\*** (informed) to move the robot through the cleaning order and back to the charging station.
4. Reports the schedule, the path of every journey, the costs, the number of expanded nodes, the final grid and the final status.

**How to run:** open this notebook in VS Code, select a Python kernel and choose **Run All**.
Keep the notebook and all input files in the same folder:
`Environment.txt`, `Schedule.txt`, `Environment_Sample.txt`, `Schedule_Sample.txt`, `Environment_Error.txt`, `Schedule_Error.txt`.
(If a file is not found next to the notebook, the notebook also looks in a sub-folder named `inputs`, in the parent folder and in the Downloads folder. The files therefore do not have to be inside an `inputs` folder.)

**Testing with other input files:** replace `Environment.txt` and `Schedule.txt` with the new files, keeping the same file names, and choose **Run All** again.
Test 1 loads whichever `Environment.txt` and `Schedule.txt` it finds, so every result is recomputed for the new files.
Tests 2 and 3 use the sample and error files. Test 4 automatically runs any other `Environment<name>.txt` + `Schedule<name>.txt` pair found in the folder, and the summary at the end lists every pair.

**Sections:** 0 Imports and configuration · 1 Input loading · 2 Graph-based search · 3 CSP scheduling · 4 Integration · 5 Test runs · 6 Results and comparative analysis · 7 Discussion

## 0. Imports and Configuration

In [1]:
import os
import re
import glob
import heapq
from collections import deque

DATA_FOLDER = "inputs"      # optional sub-folder; the notebook folder is always searched first
ALGORITHM = "A*"            # navigation algorithm used for the detailed runs: "BFS" or "A*"

## 1. Input Loading

`load_environment` reads `Environment.txt` (grid size, obstacles, dirty locations, charging station) and validates it:
the declared counts must match, every coordinate must be inside the grid, and the charger, the dirty cells and the obstacles must occupy different cells.
Dirty locations are labelled **D1, D2, …** in the order in which they appear in the file.
`unreachable_dirty` flood-fills from the charger and lists the dirty cells that cannot be reached. This is reported as a warning, so that the failure cases can still be demonstrated.

`load_schedule` reads `Schedule.txt` (number of time slots, precedence constraints, latest-slot constraints) and returns the slots together with a `constraints` dictionary:
`{"precedence": [(before, after), ...], "latest_slot": {task: slot, ...}}`.
`validate_schedule_inputs` checks that there are at least as many slots as tasks and that every task label used in a constraint is a dirty location of the environment.

`locate` finds an input file by searching the notebook folder first, then the `inputs` sub-folder, the parent folder and the Downloads folder, so the files work whether they are in an `inputs` folder or were simply downloaded next to the notebook. `find_pairs` matches each `Environment<name>.txt` with the `Schedule<name>.txt` that has the same name.

In [2]:
def parse_tuples(text):
    """'(1,3), (2,4)' -> [(1, 3), (2, 4)]"""
    return [(int(a), int(b)) for a, b in re.findall(r'\(\s*(-?\d+)\s*,\s*(-?\d+)\s*\)', text)]


def load_environment(filepath):
    """
    Load and validate an environment file.
    Returns (grid, charging_station, dirty_locations). Raises ValueError for an invalid file.

    Format (one item or group of items per line):
        rows,cols
        number of obstacles
        (r,c), (r,c), ...        (line present only if the number of obstacles > 0)
        number of dirty locations
        (r,c), (r,c), ...        (line present only if the number of dirty locations > 0)
        (r,c)                    (charging station)
    """
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    try:
        i = 0
        rows, cols = map(int, lines[i].split(',')); i += 1

        num_obstacles = int(lines[i]); i += 1
        obstacles = []
        if num_obstacles > 0:
            obstacles = parse_tuples(lines[i]); i += 1

        num_dirty = int(lines[i]); i += 1
        dirty = []
        if num_dirty > 0:
            dirty = parse_tuples(lines[i]); i += 1

        cs_list = parse_tuples(lines[i])
    except (IndexError, ValueError):
        raise ValueError("file does not follow the required Environment.txt format")

    # ---- validation ----
    if len(cs_list) != 1:
        raise ValueError("exactly one charging station is required")
    cs = cs_list[0]
    if len(obstacles) != num_obstacles:
        raise ValueError(f"declared {num_obstacles} obstacles but found {len(obstacles)}")
    if len(dirty) != num_dirty:
        raise ValueError(f"declared {num_dirty} dirty locations but found {len(dirty)}")
    all_cells = obstacles + dirty + [cs]
    for (r, c) in all_cells:
        if not (0 <= r < rows and 0 <= c < cols):
            raise ValueError(f"cell {(r, c)} is outside the {rows}x{cols} grid")
    if len(set(all_cells)) != len(all_cells):
        raise ValueError("charging station, dirty locations and obstacles must occupy different cells")

    # ---- build grid ----
    grid = [['.' for _ in range(cols)] for _ in range(rows)]
    for r, c in obstacles:
        grid[r][c] = 'X'
    for k, (r, c) in enumerate(dirty):
        grid[r][c] = f"D{k + 1}"
    grid[cs[0]][cs[1]] = 'C'
    return grid, cs, dirty


def unreachable_dirty(grid, cs, dirty):
    """Flood-fill from the charger; return the dirty locations that cannot be reached."""
    rows, cols = len(grid), len(grid[0])
    seen, queue = {cs}, deque([cs])
    while queue:
        r, c = queue.popleft()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 'X' and (nr, nc) not in seen:
                seen.add((nr, nc))
                queue.append((nr, nc))
    return [(f"D{k + 1}", d) for k, d in enumerate(dirty) if d not in seen]

In [3]:
def slot_to_int(slot):
    """'T3' -> 3"""
    return int(slot[1:])


def load_schedule(filepath):
    """
    Load a scheduling file. Returns (slots, constraints) where
        constraints = {"precedence": [(before, after), ...], "latest_slot": {task: slot, ...}}

    Format (one item per line):
        number of time slots
        number of precedence constraints
        (task_before,task_after)   x number of precedence constraints
        number of latest-slot constraints
        (task,latest_slot)         x number of latest-slot constraints
    """
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    try:
        i = 0
        num_slots = int(lines[i]); i += 1
        slots = [f"T{k + 1}" for k in range(num_slots)]

        num_prec = int(lines[i]); i += 1
        precedence = []
        for _ in range(num_prec):
            u, v = re.findall(r'[A-Za-z]\w*', lines[i].upper()); i += 1
            precedence.append((u, v))

        num_latest = int(lines[i]); i += 1
        latest_slot = {}
        for _ in range(num_latest):
            task, slot = re.findall(r'[A-Za-z]\w*', lines[i].upper()); i += 1
            # if a task has several latest-slot constraints, the tightest one applies
            if task not in latest_slot or slot_to_int(slot) < slot_to_int(latest_slot[task]):
                latest_slot[task] = slot
    except (IndexError, ValueError):
        raise ValueError("file does not follow the required Schedule.txt format")

    return slots, {"precedence": precedence, "latest_slot": latest_slot}


def validate_schedule_inputs(variables, slots, constraints):
    """Checks required by the handout. Raises ValueError describing the problem."""
    if len(slots) < len(variables):
        raise ValueError(f"{len(slots)} time slots are not enough for {len(variables)} dirty locations")
    known = set(variables)
    for u, v in constraints["precedence"]:
        for t in (u, v):
            if t not in known:
                raise ValueError(f"constraint refers to unknown task {t} (not a dirty location in the environment)")
    for t, s in constraints["latest_slot"].items():
        if t not in known:
            raise ValueError(f"constraint refers to unknown task {t} (not a dirty location in the environment)")
        if s not in slots:
            raise ValueError(f"latest-slot constraint uses unknown time slot {s}")

In [4]:
def search_folders(data_folder=DATA_FOLDER):
    """Folders searched for input files, in this order."""
    folders = [".", data_folder, ".."]
    downloads = os.path.join(os.path.expanduser("~"), "Downloads")
    if os.path.isdir(downloads):
        folders.append(downloads)
    return folders


def locate(name):
    """Path of an input file: the first folder in search_folders() that contains it."""
    for folder in search_folders():
        candidate = os.path.join(folder, name)
        if os.path.exists(candidate):
            return candidate
    return name


def resolve_folder(folder=DATA_FOLDER):
    """First folder that contains any Environment*.txt file (the notebook folder is checked first)."""
    for fld in search_folders(folder):
        if glob.glob(os.path.join(fld, "Environment*.txt")):
            return fld
    return "."

In [5]:
def find_pairs():
    """Match Environment<suffix>.txt with Schedule<suffix>.txt inside ALL searched folders."""
    pairs = []
    seen = set()
    for fld in search_folders():
        for env in sorted(glob.glob(os.path.join(fld, "Environment*.txt"))):
            suffix = os.path.basename(env)[len("Environment"):-len(".txt")]
            sched = os.path.join(fld, f"Schedule{suffix}.txt")
            if os.path.exists(sched):
                # Use absolute path to prevent running the same file twice
                abs_env = os.path.abspath(env)
                if abs_env not in seen:
                    pairs.append((env, sched))
                    seen.add(abs_env)
    return pairs

## 2. Graph-Based Search

**Problem formulation**
- **State:** the robot position `(row, col)`.
- **Initial state and goal state:** passed to `search(grid, start, goal, algorithm)`; they are not hardcoded.
- **Actions:** `UP`, `DOWN`, `LEFT`, `RIGHT`. An action is valid only if the resulting cell is inside the grid and is not an obstacle.
- **Transition model:** a valid action moves the robot to the neighbouring cell.
- **Step cost:** 1 for every move, so the path cost equals the number of moves.

**Algorithms**
- **BFS (uninformed):** expands nodes level by level. It is complete, and it is optimal here because every step costs 1.
- **A\* (informed):** always expands the node with the smallest `f(n) = g(n) + h(n)`. It is complete, and it is optimal when the heuristic is admissible.

**Heuristic:** Manhattan distance, `h(n) = |row − row_goal| + |col − col_goal|`.
- *Admissible:* with 4-directional movement, every path needs at least this many moves (obstacles can only make the real distance longer), so `h` never overestimates.
- *Consistent:* one move changes `h` by at most 1, which equals the step cost. Therefore A\* never has to re-open a node.

`nodes_expanded` counts every node once, when it is taken from the frontier. A\* ignores outdated frontier entries (a node that was already expanded), so they are not counted.
`search` returns `(path, cost, nodes_expanded)`. When no path exists, or when the start or goal is an obstacle or lies outside the grid, `path` is `None`.

In [6]:
def get_successors(state, grid):
    """Returns [(next_state, action, step_cost), ...] for valid moves (inside the grid, not an obstacle)."""
    r, c = state
    rows, cols = len(grid), len(grid[0])
    moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]
    out = []
    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 'X':
            out.append(((nr, nc), action, 1))
    return out


def goal_test(state, goal_state):
    return state == goal_state


def path_cost(path):
    """Cost = number of moves = len(path) - 1. An empty path or a single state costs 0."""
    if not path:
        return 0
    return len(path) - 1


def heuristic(state, goal_state):
    """
    Manhattan distance.
    Admissible: with 4-directional moves every path needs at least this many moves
    (obstacles can only make the real distance longer), so it never overestimates.
    Consistent: one move changes h by at most 1, which equals the step cost.
    """
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def validate_path(path, grid):
    """True if consecutive states are adjacent and never on an obstacle."""
    if not path:
        return True
    for k in range(len(path) - 1):
        r1, c1 = path[k]
        r2, c2 = path[k + 1]
        if abs(r1 - r2) + abs(c1 - c2) != 1:
            print(f"  [INVALID] Non-adjacent move {path[k]} -> {path[k + 1]}")
            return False
        if grid[r2][c2] == 'X':
            print(f"  [INVALID] Path enters an obstacle at {path[k + 1]}")
            return False
    return True


def _reconstruct(came_from, node):
    path = []
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path

In [7]:
def bfs_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded); path is None if no path exists."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = deque([start])
    came_from = {start: None}
    nodes_expanded = 0

    while frontier:
        current = frontier.popleft()
        nodes_expanded += 1
        if goal_test(current, goal):
            path = _reconstruct(came_from, current)
            return path, path_cost(path), nodes_expanded
        for nxt, _, _ in get_successors(current, grid):
            if nxt not in came_from:
                came_from[nxt] = current
                frontier.append(nxt)

    return None, 0, nodes_expanded


def a_star_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded); path is None if no path exists."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = [(heuristic(start, goal), 0, start)]      # (f, g, state)
    came_from = {start: None}
    g_costs = {start: 0}
    closed = set()
    nodes_expanded = 0

    while frontier:
        f, g_cur, current = heapq.heappop(frontier)
        if current in closed:                 # outdated queue entry: skip, it is not an expansion
            continue
        closed.add(current)
        nodes_expanded += 1
        if goal_test(current, goal):
            path = _reconstruct(came_from, current)
            return path, path_cost(path), nodes_expanded
        for nxt, _, step in get_successors(current, grid):
            new_g = g_cur + step
            if nxt not in g_costs or new_g < g_costs[nxt]:
                g_costs[nxt] = new_g
                came_from[nxt] = current
                heapq.heappush(frontier, (new_g + heuristic(nxt, goal), new_g, nxt))

    return None, 0, nodes_expanded


def _valid_cell(grid, cell):
    r, c = cell
    return 0 <= r < len(grid) and 0 <= c < len(grid[0]) and grid[r][c] != 'X'


def search(grid, start, goal, algorithm="A*"):
    """
    Dispatcher. Returns (path, cost, nodes_expanded).
    path is None when no path exists (this includes a start/goal that is an obstacle or outside the grid).
    """
    if not _valid_cell(grid, start) or not _valid_cell(grid, goal):
        return None, 0, 0
    alg = algorithm.upper().replace(" ", "")
    if alg == "BFS":
        return bfs_search(grid, start, goal)
    if alg in ("A*", "ASTAR", "A-STAR"):
        return a_star_search(grid, start, goal)
    raise ValueError(f"Unknown algorithm: {algorithm}")

## 3. Constraint Satisfaction Problem (Cleaning-Task Scheduling)

**Formulation**
- **Variables:** `D1 … Dn`, one for each dirty location in `Environment.txt`, including tasks that appear in no constraint.
- **Domains:** `{T1 … Tm}`, where `m` is the number of time slots in `Schedule.txt` (`m` must be at least the number of tasks). Domains are reduced by forward checking.
- **Constraints:**
  - *precedence:* `Da` must be cleaned before `Db`, so the slot of `Da` is smaller than the slot of `Db`;
  - *latest slot:* `Da` must be cleaned no later than `Tk`;
  - *slot exclusivity:* at most one task per time slot. It applies to all tasks automatically and is not listed in the file.

All functions receive the same `constraints` dictionary (`{"precedence": [...], "latest_slot": {...}}`).

**Functions**
- `select_unassigned_variable(assignment, domains, constraints)`: MRV (the variable with the fewest remaining values). Ties are broken by the degree heuristic: the variable that shares precedence constraints with the largest number of *unassigned* variables.
- `order_domain_values(variable, assignment, domains, constraints)`: Least Constraining Value. The slots that remove the fewest options from the other unassigned variables are tried first.
- `is_consistent(variable, value, assignment, constraints)`: checks slot exclusivity, the latest-slot constraint and the precedence constraints against the current partial assignment.
- `forward_check(variable, value, assignment, domains, constraints)`: removes infeasible slots from the domains of the unassigned variables. It returns `None` if any domain becomes empty. It works on a copy of the domains, so the removed values are restored automatically when the search backtracks.
- `backtracking_search(variables, domains, constraints)`: the recursive solver that uses all of the above. It returns a complete assignment `{task: slot}`, or `None` if no feasible schedule exists.
- `verify_schedule(solution, slots, constraints)`: checks independently that the generated schedule satisfies every constraint and prints a PASS or FAIL line for each check.

In [8]:
def is_consistent(variable, value, assignment, constraints):
    """True if variable=value violates no constraint that can be evaluated with the current assignment."""
    precedence, latest_slot = constraints["precedence"], constraints["latest_slot"]
    # 1. slot exclusivity
    if value in assignment.values():
        return False
    # 2. latest slot
    if variable in latest_slot and slot_to_int(value) > slot_to_int(latest_slot[variable]):
        return False
    # 3. precedence
    for u, v in precedence:
        if v == variable and u in assignment and slot_to_int(assignment[u]) >= slot_to_int(value):
            return False
        if u == variable and v in assignment and slot_to_int(value) >= slot_to_int(assignment[v]):
            return False
    return True


def forward_check(variable, value, assignment, domains, constraints):
    """
    Prune the domains of the unassigned variables after variable=value.
    Returns the new domains, or None if any unassigned variable ends up with an empty domain.
    It works on a copy, so the removed values are restored automatically when the search backtracks.
    """
    precedence, latest_slot = constraints["precedence"], constraints["latest_slot"]
    new_domains = {v: list(d) for v, d in domains.items()}
    for other in domains:
        if other in assignment:
            continue
        # slot exclusivity
        if value in new_domains[other]:
            new_domains[other].remove(value)
        # latest slot
        if other in latest_slot:
            new_domains[other] = [s for s in new_domains[other]
                                  if slot_to_int(s) <= slot_to_int(latest_slot[other])]
        # precedence
        for u, v in precedence:
            if variable == u and other == v:
                new_domains[other] = [s for s in new_domains[other] if slot_to_int(s) > slot_to_int(value)]
            if variable == v and other == u:
                new_domains[other] = [s for s in new_domains[other] if slot_to_int(s) < slot_to_int(value)]
        if not new_domains[other]:
            return None
    return new_domains


def select_unassigned_variable(assignment, domains, constraints):
    """MRV (fewest remaining values); ties are broken by the degree heuristic (most unassigned neighbours)."""
    precedence = constraints["precedence"]
    unassigned = [v for v in domains if v not in assignment]
    if not unassigned:
        return None
    min_len = min(len(domains[v]) for v in unassigned)
    candidates = [v for v in unassigned if len(domains[v]) == min_len]
    if len(candidates) == 1:
        return candidates[0]

    def degree(var):
        neighbours = set()
        for u, v in precedence:
            if u == var and v not in assignment:
                neighbours.add(v)
            if v == var and u not in assignment:
                neighbours.add(u)
        return len(neighbours)

    return max(candidates, key=degree)


def order_domain_values(variable, assignment, domains, constraints):
    """Least Constraining Value: the values that remove the fewest options from other variables come first."""
    precedence, latest_slot = constraints["precedence"], constraints["latest_slot"]

    def eliminated(value):
        count = 0
        for other in domains:
            if other == variable or other in assignment:
                continue
            for val in domains[other]:
                if val == value:                                   # slot exclusivity
                    count += 1
                    continue
                if other in latest_slot and slot_to_int(val) > slot_to_int(latest_slot[other]):
                    count += 1                                     # already ruled out by a latest-slot constraint
                    continue
                for u, v in precedence:                            # precedence
                    if variable == u and other == v and slot_to_int(val) <= slot_to_int(value):
                        count += 1
                        break
                    if variable == v and other == u and slot_to_int(val) >= slot_to_int(value):
                        count += 1
                        break
        return count

    return sorted(domains[variable], key=lambda val: (eliminated(val), slot_to_int(val)))

In [9]:
def backtracking_search(variables, domains, constraints):
    """Returns a complete assignment {task: slot}, or None if no feasible schedule exists."""
    def backtrack(assignment, cur_domains):
        if len(assignment) == len(variables):
            return dict(assignment)
        var = select_unassigned_variable(assignment, cur_domains, constraints)
        for value in order_domain_values(var, assignment, cur_domains, constraints):
            if is_consistent(var, value, assignment, constraints):
                assignment[var] = value
                nd = forward_check(var, value, assignment, cur_domains, constraints)
                if nd is not None:
                    result = backtrack(assignment, nd)
                    if result is not None:
                        return result
                del assignment[var]                      # backtrack
        return None

    return backtrack({}, domains)


def verify_schedule(solution, slots, constraints, verbose=True):
    """Independent check that a schedule satisfies every constraint. Returns True if all checks pass."""
    all_ok = True

    def report(message, passed):
        nonlocal all_ok
        all_ok = all_ok and passed
        if verbose:
            print(f"    [{'PASS' if passed else 'FAIL'}] {message}")

    report("every task has exactly one time slot", all(s in slots for s in solution.values()))
    report("no two tasks share a time slot", len(set(solution.values())) == len(solution))
    for u, v in constraints["precedence"]:
        report(f"{u} before {v}  ({solution[u]} < {solution[v]})",
               slot_to_int(solution[u]) < slot_to_int(solution[v]))
    for t, s in constraints["latest_slot"].items():
        report(f"{t} no later than {s}  (assigned {solution[t]})",
               slot_to_int(solution[t]) <= slot_to_int(s))
    return all_ok

## 4. Integration

`execute_schedule(grid, charging_station, schedule, algorithm)` receives the cleaning tasks in the order given by the CSP, as `(label, (row, col))` pairs. It then:
1. starts at the charging station;
2. calls `search` from the robot's current position to each cleaning location in the scheduled order;
3. appends every returned path to the complete path without repeating the shared end point, and marks the task as cleaned when the robot reaches it;
4. calls `search` once more to bring the robot back to the charging station;
5. returns `(complete_path, total_cost, success)`.

If a required path cannot be found, it stops and returns `success = False`, so it never reports an unfinished task as completed.

`run_pipeline` connects the components (load the files, run the CSP, order the tasks, navigate) and prints the schedule, the constraint checks, the visited sequence, the path of every journey, the final grid, the total movements and cost, and the final status.
`compare_algorithms` solves the same journeys with BFS and with A\* and compares paths, costs and expanded nodes journey by journey and in total.

The final grid uses these symbols: `C` charger, `X` obstacle, `*` cells on the robot path, lowercase `d1, d2, …` cleaned dirty cells, and uppercase `D1, D2, …` dirty cells that were not cleaned.

In [10]:
def _execute(grid, charging_station, schedule, algorithm):
    """
    Core routine. Returns (complete_path, total_cost, success, legs, cleaned),
    where legs is a list of per-journey records (used for reporting).
    """
    current = charging_station
    complete_path = [charging_station]
    legs, cleaned = [], []
    targets = list(schedule) + [("Charger", charging_station)]

    for label, goal in targets:
        path, cost, nodes = search(grid, current, goal, algorithm)
        legs.append({'from': current, 'to': goal, 'label': label, 'path': path, 'cost': cost, 'nodes': nodes})

        if path is None:                                   # required path not found -> report failure
            return complete_path, path_cost(complete_path), False, legs, cleaned
        if not validate_path(path, grid):
            legs[-1]['path'] = None
            return complete_path, path_cost(complete_path), False, legs, cleaned

        complete_path.extend(path[1:])                     # skip the duplicated shared endpoint
        current = goal
        if label != "Charger":
            cleaned.append(label)                          # robot reached it -> mark as cleaned

    total_cost = path_cost(complete_path)
    return complete_path, total_cost, True, legs, cleaned


def execute_schedule(grid, charging_station, schedule, algorithm="A*"):
    """
    Executes the CSP-generated schedule using graph-based search.

    schedule : cleaning tasks in slot order, as (label, (row, col)), e.g. [("D3", (4, 1)), ("D1", (2, 4))]
    Returns  : (complete_path, total_cost, success)
    """
    complete_path, total_cost, success, _, _ = _execute(grid, charging_station, schedule, algorithm)
    return complete_path, total_cost, success


def print_grid(grid, title=""):
    if title:
        print(f"  {title}")
    for row in grid:
        print("    " + "  ".join(f"{cell:>2}" for cell in row))


def print_final_grid(grid, cs, complete_path, cleaned):
    """Final grid: robot path marked with *, cleaned dirty cells in lowercase, uncleaned ones unchanged."""
    visited = set(complete_path)
    out = []
    for r in range(len(grid)):
        row = []
        for c in range(len(grid[0])):
            cell = grid[r][c]
            if cell == 'X':
                sym = 'X'
            elif (r, c) == cs:
                sym = 'C'
            elif cell.startswith('D'):
                sym = cell.lower() if cell in cleaned else cell
            elif (r, c) in visited:
                sym = '*'
            else:
                sym = '.'
            row.append(sym)
        out.append(row)
    print_grid(out)
    print("    Legend: C charger | X obstacle | * robot path | d1,d2.. cleaned | D1,D2.. NOT cleaned | . untouched")

In [11]:
def run_pipeline(env_file, schedule_file, algorithm="A*", verbose=True):
    """Complete pipeline for one (environment, schedule) pair. Returns a result dictionary."""
    def say(*args):
        if verbose:
            print(*args)

    say("=" * 70)
    say(f"  PIPELINE: {os.path.basename(env_file)}  +  {os.path.basename(schedule_file)}   [{algorithm}]")
    say("=" * 70)

    # ---- 1. load and validate the inputs ----
    try:
        grid, cs, dirty = load_environment(env_file)
        slots, constraints = load_schedule(schedule_file)
        dirty_map = {f"D{k + 1}": d for k, d in enumerate(dirty)}
        variables = list(dirty_map)
        validate_schedule_inputs(variables, slots, constraints)
    except FileNotFoundError as e:
        say(f"  [ERROR] File not found: {e.filename}")
        return {"success": False, "reason": f"file not found: {e.filename}"}
    except ValueError as e:
        say(f"  [ERROR] Invalid input: {e}")
        return {"success": False, "reason": f"invalid input: {e}"}

    say(f"  Charging station : {cs}")
    say(f"  Dirty locations  : {dirty_map}")
    say(f"  Time slots       : {slots}")
    say(f"  Precedence       : {constraints['precedence']}")
    say(f"  Latest slots     : {constraints['latest_slot']}")
    unreachable = unreachable_dirty(grid, cs, dirty)
    if unreachable:
        say(f"  [WARNING] Unreachable dirty locations: {unreachable}")
    say()
    if verbose:
        print_grid(grid, "BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):")
    say()

    # ---- 2. CSP: generate the cleaning schedule ----
    domains = {v: list(slots) for v in variables}
    solution = backtracking_search(variables, domains, constraints)
    if solution is None:
        say("  [FAILURE] The CSP found no feasible schedule: the constraints contradict each other.")
        say("  STATUS: FAILURE - the robot does not move; no dirty location was cleaned.")
        say()
        return {"success": False, "reason": "no feasible schedule exists"}

    order = sorted(solution, key=lambda t: slot_to_int(solution[t]))
    schedule = [(t, dirty_map[t]) for t in order]
    say("  Cleaning schedule generated by the CSP:")
    for t in order:
        say(f"    {solution[t]} -> {t} at {dirty_map[t]}")
    say("  Constraint verification:")
    if verbose:
        verify_schedule(solution, slots, constraints)
    say()

    # ---- 3. navigation ----
    path, cost, ok, legs, cleaned = _execute(grid, cs, schedule, algorithm)
    say(f"  Dirty locations visited, in order: {' -> '.join(cleaned) if cleaned else '(none)'}")
    say()
    say("  Journeys:")
    total_nodes = 0
    for leg in legs:
        total_nodes += leg['nodes']
        if leg['path'] is None:
            say(f"    {leg['from']} -> {leg['label']} {leg['to']}: NO PATH FOUND (nodes expanded: {leg['nodes']})")
        else:
            say(f"    {leg['from']} -> {leg['label']} {leg['to']}: cost {leg['cost']}, nodes expanded {leg['nodes']}")
            say(f"       path: {leg['path']}")
    say()
    if verbose:
        print("  Final grid:")
        print_final_grid(grid, cs, path, cleaned)
    say()
    say(f"  Complete path    : {path}")
    say(f"  Total movements  : {len(path) - 1}")
    say(f"  Total cost       : {cost}")
    say(f"  Nodes expanded   : {total_nodes}")
    if ok:
        say("  STATUS: SUCCESS - all dirty locations were cleaned and the robot returned to the charging station.")
    else:
        missing = [t for t in order if t not in cleaned]
        say(f"  STATUS: FAILURE - the task was NOT completed. Cleaned: {cleaned if cleaned else 'none'} | Not cleaned: {missing}")
    say()

    return {"success": ok, "reason": "OK" if ok else "a required path was not found",
            "solution": solution, "schedule": schedule, "path": path, "cost": cost,
            "moves": len(path) - 1, "nodes": total_nodes, "cleaned": cleaned}


def compare_algorithms(env_file, schedule_file):
    """
    Compare BFS and A* on the same journeys (same start and goal for both algorithms):
    path, cost and expanded nodes for every journey, then the totals.
    """
    print("=" * 70)
    print(f"  ALGORITHM COMPARISON: {os.path.basename(env_file)} + {os.path.basename(schedule_file)}")
    print("=" * 70)
    try:
        grid, cs, dirty = load_environment(env_file)
        slots, constraints = load_schedule(schedule_file)
        dirty_map = {f"D{k + 1}": d for k, d in enumerate(dirty)}
        variables = list(dirty_map)
        validate_schedule_inputs(variables, slots, constraints)
    except FileNotFoundError as e:
        print(f"  [ERROR] File not found: {e.filename}\n")
        return None
    except ValueError as e:
        print(f"  [ERROR] Invalid input: {e}\n")
        return None

    solution = backtracking_search(variables, {v: list(slots) for v in variables}, constraints)
    if solution is None:
        print("  No feasible schedule exists, so there are no journeys to compare.\n")
        return None

    order = sorted(solution, key=lambda t: slot_to_int(solution[t]))
    targets = [(t, dirty_map[t]) for t in order] + [("Charger", cs)]
    totals = {"BFS": {"cost": 0, "nodes": 0}, "A*": {"cost": 0, "nodes": 0}}
    same_cost, completed, same_path, current = True, True, True, cs

    for k, (label, goal) in enumerate(targets, start=1):
        print(f"  Journey {k}: {current} -> {label} {goal}")
        found = {alg: search(grid, current, goal, alg) for alg in ("BFS", "A*")}
        for alg in found:
            totals[alg]["nodes"] += found[alg][2]
        if any(found[alg][0] is None for alg in found):
            for alg in found:
                print(f"    {alg:<4} NO PATH FOUND (nodes expanded: {found[alg][2]})")
            print(f"  The comparison stops here: journey {k} is impossible.")
            completed = False
            break
        for alg in ("BFS", "A*"):
            path, cost, nodes = found[alg]
            totals[alg]["cost"] += cost
            print(f"    {alg:<4} cost {cost:<3} nodes expanded {nodes:<4} path: {path}")
        same_cost = same_cost and found["BFS"][1] == found["A*"][1]
        same_path = same_path and found["BFS"][0] == found["A*"][0]
        current = goal

    print()
    print(f"  {'Algorithm':<12}{'Total cost':<14}{'Nodes expanded':<16}")
    print("  " + "-" * 42)
    for alg in ("BFS", "A*"):
        print(f"  {alg:<12}{totals[alg]['cost']:<14}{totals[alg]['nodes']:<16}")
    if totals["BFS"]["nodes"] > 0:
        saved = 100 * (1 - totals["A*"]["nodes"] / totals["BFS"]["nodes"])
        if totals["A*"]["nodes"] == totals["BFS"]["nodes"]:
            print("  A* and BFS expanded the same number of nodes.")
        else:
            print(f"  A* expanded {abs(saved):.1f}% {'fewer' if saved > 0 else 'more'} nodes than BFS.")
    if completed:
        print(f"  Same cost on every journey : {'yes' if same_cost else 'NO'}")
        print(f"  Identical paths            : {'yes' if same_path else 'no (equal-cost alternatives)'}")
    else:
        print("  Note: the totals cover only the journeys before the impossible one (the impossible journey adds only its expanded nodes).")
    print()
    return {"totals": totals, "completed": completed, "same_cost": same_cost}

## 5. Test Runs

Two standalone tests (Tests 0a and 0b) first show that the search component and the CSP component work on their own.
Tests 1 to 4 then run the complete pipeline with A\* (schedule, constraint checks, visited sequence, path of every journey, final grid, total cost, final status) and compare BFS and A\* journey by journey.

In [12]:
def f(name):
    # path of an input file: next to the notebook first, then inside the data folder
    return locate(name)

required = ["Environment.txt", "Schedule.txt", "Environment_Sample.txt", "Schedule_Sample.txt",
            "Environment_Error.txt", "Schedule_Error.txt"]
missing = [n for n in required if not os.path.exists(f(n))]
print("Input files:")
for n in required:
    print(f"   {n:<24}", "found" if n not in missing else "MISSING")
if missing:
    print("\n[WARNING] Some input files were not found. Put them in the same folder as this notebook:")
    print("   notebook folder =", os.path.abspath("."))
else:
    print("\nAll required input files were found.")
print("\nInput files used by Test 1:", f("Environment.txt"), "and", f("Schedule.txt"))

Input files:
   Environment.txt          found
   Schedule.txt             found
   Environment_Sample.txt   found
   Schedule_Sample.txt      found
   Environment_Error.txt    found
   Schedule_Error.txt       found

All required input files were found.

Input files used by Test 1: .\Environment.txt and .\Schedule.txt


### Test 0a — Search component on its own (edge cases)
Start equal to the goal, an invalid start (an obstacle), a goal outside the grid, `path_cost` of an empty path and of a one-state path, and an unreachable goal (D1 is walled in inside `Environment_Error.txt`).

In [13]:
if not os.path.exists(f("Environment.txt")):
    print("Environment.txt was not found, so Test 0a was skipped.")
else:
    grid, cs, dirty = load_environment(f("Environment.txt"))
    obstacle = next(((r, c) for r in range(len(grid)) for c in range(len(grid[0])) if grid[r][c] == "X"), None)
    print("start == goal         :", search(grid, cs, cs, "BFS"), "|", search(grid, cs, cs, "A*"))
    if obstacle is not None:
        print("invalid start (wall)  :", search(grid, obstacle, dirty[0], "A*"))
    print("goal outside the grid :", search(grid, cs, (99, 99), "BFS"))
    print("path_cost([]) =", path_cost([]), "| path_cost([(0, 0)]) =", path_cost([(0, 0)]))

    if os.path.exists(f("Environment_Error.txt")):
        egrid, ecs, edirty = load_environment(f("Environment_Error.txt"))
        for alg in ("BFS", "A*"):
            path, cost, nodes = search(egrid, ecs, edirty[0], alg)
            print(f"unreachable D1 {edirty[0]} with {alg:<3}: path={path}, cost={cost}, nodes expanded={nodes}")
    else:
        print("Environment_Error.txt was not found, so the unreachable-goal check was skipped.")

start == goal         : ([(3, 0)], 0, 0) | ([(3, 0)], 0, 0)
invalid start (wall)  : (None, 0, 0)
goal outside the grid : (None, 0, 0)
path_cost([]) = 0 | path_cost([(0, 0)]) = 0
unreachable D1 (2, 4) with BFS: path=None, cost=0, nodes expanded=53
unreachable D1 (2, 4) with A* : path=None, cost=0, nodes expanded=53


### Test 0b — CSP component on its own
The CSP is solved without any navigation. The variables come from the dirty locations of the matching environment file, and the generated schedule is checked against every constraint.

In [14]:
def test_csp(env_file, schedule_file):
    _, _, dirty = load_environment(env_file)
    variables = [f"D{k + 1}" for k in range(len(dirty))]
    slots, constraints = load_schedule(schedule_file)
    print(f"--- CSP only: {os.path.basename(schedule_file)} (tasks from {os.path.basename(env_file)}: {variables}) ---")
    print("  Slots        :", slots)
    print("  Precedence   :", constraints["precedence"])
    print("  Latest slots :", constraints["latest_slot"])
    validate_schedule_inputs(variables, slots, constraints)
    solution = backtracking_search(variables, {v: list(slots) for v in variables}, constraints)
    if solution is None:
        print("  [FAILURE] No feasible schedule exists.\n")
        return
    print("  Schedule     :", {s: t for t, s in sorted(solution.items(), key=lambda kv: slot_to_int(kv[1]))})
    print("  Constraint verification:")
    verify_schedule(solution, slots, constraints)
    print()

for env_name, sched_name in [("Environment.txt", "Schedule.txt"),
                             ("Environment_Sample.txt", "Schedule_Sample.txt"),
                             ("Environment_Error.txt", "Schedule_Error.txt")]:
    if os.path.exists(f(env_name)) and os.path.exists(f(sched_name)):
        try:
            test_csp(f(env_name), f(sched_name))
        except ValueError as e:
            print(f"--- CSP only: {sched_name} ---\n  [ERROR] Invalid input: {e}\n")
    else:
        print(f"{env_name} / {sched_name}: file not found, skipped.\n")

--- CSP only: Schedule.txt (tasks from Environment.txt: ['D1', 'D2', 'D3', 'D4']) ---
  Slots        : ['T1', 'T2', 'T3', 'T4']
  Precedence   : [('D3', 'D1'), ('D1', 'D2'), ('D3', 'D4')]
  Latest slots : {'D2': 'T3', 'D4': 'T4'}
  Schedule     : {'T1': 'D3', 'T2': 'D1', 'T3': 'D2', 'T4': 'D4'}
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D3 before D1  (T1 < T2)
    [PASS] D1 before D2  (T2 < T3)
    [PASS] D3 before D4  (T1 < T4)
    [PASS] D2 no later than T3  (assigned T3)
    [PASS] D4 no later than T4  (assigned T4)

--- CSP only: Schedule_Sample.txt (tasks from Environment_Sample.txt: ['D1', 'D2', 'D3', 'D4']) ---
  Slots        : ['T1', 'T2', 'T3', 'T4']
  Precedence   : [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D4')]
  Latest slots : {'D3': 'T3', 'D2': 'T3'}
  Schedule     : {'T1': 'D1', 'T2': 'D2', 'T3': 'D3', 'T4': 'D4'}
  Constraint verification:
    [PASS] every task has exactly one time slot
    

### Test 1 — Own environment (`Environment.txt` + `Schedule.txt`)

In [15]:
run_pipeline(f("Environment.txt"), f("Schedule.txt"), ALGORITHM)
compare_algorithms(f("Environment.txt"), f("Schedule.txt"))

  PIPELINE: Environment.txt  +  Schedule.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D3', 'D1'), ('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T3', 'D4': 'T4'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   .   .   X   .
     .   .   .   .  D1   .   .   .
     C   .   .   .   .   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  Cleaning schedule generated by the CSP:
    T1 -> D3 at (4, 1)
    T2 -> D1 at (2, 4)
    T3 -> D2 at (5, 6)
    T4 -> D4 at (6, 3)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D3 before D1  (T1 < T2)
    [PASS] D1 before D2  (T2 < T3)
    [PASS] D3 before

{'totals': {'BFS': {'cost': 22, 'nodes': 138},
  'A*': {'cost': 22, 'nodes': 46}},
 'completed': True,
 'same_cost': True}

### Test 2 — Provided sample (`Environment_Sample.txt` + `Schedule_Sample.txt`)

In [16]:
run_pipeline(f("Environment_Sample.txt"), f("Schedule_Sample.txt"), ALGORITHM)
compare_algorithms(f("Environment_Sample.txt"), f("Schedule_Sample.txt"))

  PIPELINE: Environment_Sample.txt  +  Schedule_Sample.txt   [A*]
  Charging station : (0, 0)
  Dirty locations  : {'D1': (0, 7), 'D2': (3, 7), 'D3': (7, 7), 'D4': (7, 0)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D4')]
  Latest slots     : {'D3': 'T3', 'D2': 'T3'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     C   .   .   .   .   .   .  D1
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .  D2
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
     .   .   .   X   .   .   .   .
    D4   .   .   .   .   .   .  D3

  Cleaning schedule generated by the CSP:
    T1 -> D1 at (0, 7)
    T2 -> D2 at (3, 7)
    T3 -> D3 at (7, 7)
    T4 -> D4 at (7, 0)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D1 before D3  (T1 < T3)
    [PASS] D2 before D4  (T2 < T4)
    [P

{'totals': {'BFS': {'cost': 28, 'nodes': 118},
  'A*': {'cost': 28, 'nodes': 33}},
 'completed': True,
 'same_cost': True}

### Test 3 — Failure cases

**Test 3a: infeasible schedule** (`Environment_Error.txt` + `Schedule_Error.txt`).
The precedence constraints form a cycle (D1 before D2, D2 before D3, D3 before D1), so no feasible schedule exists. The CSP reports the failure and the robot does not move.

In [17]:
run_pipeline(f("Environment_Error.txt"), f("Schedule_Error.txt"), ALGORITHM)
compare_algorithms(f("Environment_Error.txt"), f("Schedule_Error.txt"))

  PIPELINE: Environment_Error.txt  +  Schedule_Error.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D1', 'D2'), ('D2', 'D3'), ('D3', 'D1')]
  Latest slots     : {'D4': 'T1', 'D2': 'T3'}
  [WARNING] Unreachable dirty locations: [('D1', (2, 4))]

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   X   .   X   .
     .   .   .   X  D1   X   .   .
     C   .   .   .   X   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  [FAILURE] The CSP found no feasible schedule: the constraints contradict each other.
  STATUS: FAILURE - the robot does not move; no dirty location was cleaned.

  ALGORITHM COMPARISON: Environment_Error.txt + Schedule_Error.txt
  No feasible schedule exists, so there 

**Test 3b: unreachable destination** (`Environment_Error.txt` + `Schedule_Sample.txt`).
The schedule is feasible, but D1 is surrounded by obstacles in `Environment_Error.txt`. The graph search cannot reach it, so the integrated run reports a failure and lists which tasks were and were not cleaned. It does not claim that the cleaning was completed.

In [18]:
run_pipeline(f("Environment_Error.txt"), f("Schedule_Sample.txt"), ALGORITHM)
compare_algorithms(f("Environment_Error.txt"), f("Schedule_Sample.txt"))

  PIPELINE: Environment_Error.txt  +  Schedule_Sample.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : {'D1': (2, 4), 'D2': (5, 6), 'D3': (4, 1), 'D4': (6, 3)}
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D4')]
  Latest slots     : {'D3': 'T3', 'D2': 'T3'}
  [WARNING] Unreachable dirty locations: [('D1', (2, 4))]

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
     .   .   .   .   .   .   .   .
     .   .   .   X   X   .   X   .
     .   .   .   X  D1   X   .   .
     C   .   .   .   X   .   .   .
     .  D3   .   .   X   .   .   .
     .   .   X   .   .   .  D2   .
     X   .   .  D4   .   X   .   .
     .   .   .   .   .   .   .   .

  Cleaning schedule generated by the CSP:
    T1 -> D1 at (2, 4)
    T2 -> D2 at (5, 6)
    T3 -> D3 at (4, 1)
    T4 -> D4 at (6, 3)
  Constraint verification:
    [PASS] every task has exactly one time slot
    [PASS] no two tasks share a time slot
    [PASS] D1 befor

{'totals': {'BFS': {'cost': 0, 'nodes': 53}, 'A*': {'cost': 0, 'nodes': 53}},
 'completed': False,
 'same_cost': True}

### Test 4 — Any other input pairs in the folder
Every other `Environment<name>.txt` + `Schedule<name>.txt` pair found next to the notebook is run automatically (the names must match, for example `Environment_Test1.txt` with `Schedule_Test1.txt`).
Files that have no matching partner are reported. The pairs used by Tests 1 to 3 are not repeated.

In [19]:
standard = {"Environment.txt", "Environment_Sample.txt", "Environment_Error.txt"}

def file_names(prefix):
    names = set()
    for fld in search_folders():
        for n in glob.glob(os.path.join(fld, prefix + "*.txt")):
            names.add(os.path.basename(n)[len(prefix):-len(".txt")])
    return names

for name in sorted(file_names("Environment") - file_names("Schedule")):
    print(f"[WARNING] Environment{name}.txt has no matching Schedule{name}.txt, so it was not run.")
for name in sorted(file_names("Schedule") - file_names("Environment")):
    print(f"[WARNING] Schedule{name}.txt has no matching Environment{name}.txt, so it was not run.")

# Yahan find_pairs() se brackets ke andar 'folder' hata diya gaya hai
extra_pairs = [(e, s) for e, s in find_pairs() if os.path.basename(e) not in standard]
if not extra_pairs:
    print("No additional Environment/Schedule pairs were found. Tests 1 to 3 cover all files in the folder.")
for e, s in extra_pairs:
    run_pipeline(e, s, ALGORITHM)
    compare_algorithms(e, s)

  PIPELINE: Environment_Test1.txt  +  Schedule_Test1.txt   [A*]
  Charging station : (4, 4)
  Dirty locations  : {'D1': (0, 9), 'D2': (5, 5), 'D3': (9, 0), 'D4': (0, 0), 'D5': (9, 9)}
  Time slots       : ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
  Precedence       : [('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T4', 'D5': 'T6'}

  BEFORE cleaning (C = charger, D1..Dn = dirty, X = obstacle):
    D4   .   .   .   .   .   .   .   .  D1
     .   .   .   .   .   .   .   .   .   .
     .   .   X   .   .   .   .   .   .   .
     .   .   .   .   .   .   .   .   .   .
     .   .   .   .   C   .   .   .   .   .
     .   .   .   .   .  D2   .   .   .   .
     .   .   .   .   .   .   .   .   .   .
     .   .   .   .   .   .   .   X   .   .
     .   .   .   .   .   .   .   .   .   .
    D3   .   .   .   .   .   .   .   .  D5

  Cleaning schedule generated by the CSP:
    T1 -> D1 at (0, 9)
    T2 -> D2 at (5, 5)
    T3 -> D3 at (9, 0)
    T4 -> D4 at (0, 0)
    T5 -> D5 at (9, 9)
  Constrai

### Optional tool — interactive mode (file picker)
Set `INTERACTIVE_MODE = True` and run the next cell to choose any `Environment*.txt` file in a dialog window.
The matching `Schedule*.txt` (same name after the prefix, for example `Environment_Test.txt` and `Schedule_Test.txt`) is found automatically.
It is switched off by default so that **Run All** never waits for a pop-up window.

In [20]:
INTERACTIVE_MODE = False     # True -> open a file dialog to choose an Environment file

if INTERACTIVE_MODE:
    selected_env = ""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()                         # hide the empty main window
        root.attributes('-topmost', True)       # keep the dialog on top of other windows
        root.update()
        selected_env = filedialog.askopenfilename(
            title="Select Environment File",
            initialdir=os.path.abspath(resolve_folder()),
            filetypes=[("Text Files", "*.txt")])
        root.destroy()
    except Exception as e:
        print("The file dialog is not available here:", e)

    if not selected_env:
        print("No file selected (cancelled).")
    else:
        env_name = os.path.basename(selected_env)
        if not (env_name.startswith("Environment") and env_name.endswith(".txt")):
            print("Please select a file named Environment*.txt")
        else:
            suffix = env_name[len("Environment"):-len(".txt")]
            selected_sched = os.path.join(os.path.dirname(selected_env), f"Schedule{suffix}.txt")
            print("Selected environment :", env_name)
            print("Matching schedule    :", os.path.basename(selected_sched))
            if not os.path.exists(selected_sched):
                print("[ERROR] The matching schedule file was not found. Both files must share the same name after the prefix.")
            else:
                run_pipeline(selected_env, selected_sched, ALGORITHM)
                compare_algorithms(selected_env, selected_sched)
else:
    print("Interactive mode is off. Set INTERACTIVE_MODE = True to choose an Environment file in a dialog.")

Interactive mode is off. Set INTERACTIVE_MODE = True to choose an Environment file in a dialog.


## 6. Results and Comparative Analysis

The table lists every Environment/Schedule pair found in the folder, plus the mixed failure case of Test 3b, for both algorithms.
The statements below the table are generated from the measured results, so they always describe the files that were actually run.

In [21]:
all_runs = list(find_pairs())
mixed = (f("Environment_Error.txt"), f("Schedule_Sample.txt"))
if os.path.exists(mixed[0]) and os.path.exists(mixed[1]) and mixed not in all_runs:
    all_runs.append(mixed)

results = {}                                   # pair name -> {"BFS": result, "A*": result}
for e, s in all_runs:
    name = f"{os.path.basename(e)} + {os.path.basename(s)}"
    results[name] = {alg: run_pipeline(e, s, alg, verbose=False) for alg in ("BFS", "A*")}

print(f"{'Pair':<48}{'Algo':<6}{'Success':<9}{'Cost':<7}{'Nodes':<7}Note")
print("-" * 100)
for name, res in results.items():
    for alg in ("BFS", "A*"):
        r = res[alg]
        if "cost" not in r:
            print(f"{name:<48}{alg:<6}{'False':<9}{'-':<7}{'-':<7}{r['reason']}")
            continue
        note = "" if r["success"] else r["reason"]
        if alg == "A*" and "nodes" in res["BFS"] and res["BFS"]["nodes"] > 0:
            saved = 100 * (1 - r["nodes"] / res["BFS"]["nodes"])
            extra = "same nodes as BFS" if r["nodes"] == res["BFS"]["nodes"] else f"{abs(saved):.1f}% {'fewer' if saved > 0 else 'more'} nodes than BFS"
            note = (note + "; " if note else "") + extra
        print(f"{name:<48}{alg:<6}{str(r['success']):<9}{r['cost']:<7}{r['nodes']:<7}{note}")

Pair                                            Algo  Success  Cost   Nodes  Note
----------------------------------------------------------------------------------------------------
Environment.txt + Schedule.txt                  BFS   True     22     138    
Environment.txt + Schedule.txt                  A*    True     22     46     66.7% fewer nodes than BFS
Environment_Error.txt + Schedule_Error.txt      BFS   False    -      -      no feasible schedule exists
Environment_Error.txt + Schedule_Error.txt      A*    False    -      -      no feasible schedule exists
Environment_Sample.txt + Schedule_Sample.txt    BFS   True     28     118    
Environment_Sample.txt + Schedule_Sample.txt    A*    True     28     33     72.0% fewer nodes than BFS
Environment_Test1.txt + Schedule_Test1.txt      BFS   True     64     443    
Environment_Test1.txt + Schedule_Test1.txt      A*    True     64     233    47.4% fewer nodes than BFS
Environment_Test2.txt + Schedule_Test2.txt      BFS   True   

In [22]:
print("Analysis of the measured results\n")
for name, res in results.items():
    b, a = res["BFS"], res["A*"]
    if "cost" not in b:
        print(f"- {name}: not run to navigation ({b['reason']}).")
    elif b["success"] and a["success"]:
        saved = 100 * (1 - a["nodes"] / b["nodes"]) if b["nodes"] else 0.0
        same = "the same total cost" if a["cost"] == b["cost"] else "DIFFERENT total costs"
        print(f"- {name}: both algorithms completed the task with {same} ({b['cost']} moves). "
              f"A* expanded {a['nodes']} nodes and BFS expanded {b['nodes']} ({saved:.1f}% fewer for A*).")
    else:
        print(f"- {name}: navigation failed ({b['reason']}). Cleaned before the failure: "
              f"{b['cleaned'] if b['cleaned'] else 'none'}. A* expanded {a['nodes']} nodes and BFS expanded {b['nodes']}.")

Analysis of the measured results

- Environment.txt + Schedule.txt: both algorithms completed the task with the same total cost (22 moves). A* expanded 46 nodes and BFS expanded 138 (66.7% fewer for A*).
- Environment_Error.txt + Schedule_Error.txt: not run to navigation (no feasible schedule exists).
- Environment_Sample.txt + Schedule_Sample.txt: both algorithms completed the task with the same total cost (28 moves). A* expanded 33 nodes and BFS expanded 118 (72.0% fewer for A*).
- Environment_Test1.txt + Schedule_Test1.txt: both algorithms completed the task with the same total cost (64 moves). A* expanded 233 nodes and BFS expanded 443 (47.4% fewer for A*).
- Environment_Test2.txt + Schedule_Test2.txt: both algorithms completed the task with the same total cost (40 moves). A* expanded 111 nodes and BFS expanded 171 (35.1% fewer for A*).
- Environment_Test3.txt + Schedule_Test3.txt: not run to navigation (no feasible schedule exists).
- Environment_Test4.txt + Schedule_Test4.txt: na

## 7. Discussion

- **Correctness:** every returned path is checked with `validate_path` (adjacent cells, no obstacle cells), and every schedule is checked with `verify_schedule`. Failures are reported explicitly: the CSP reports an infeasible schedule, and `execute_schedule` reports a missing path with `success = False` instead of claiming that the cleaning was completed.
- **BFS:** complete, and optimal for unit step costs. It is uninformed, so it expands nodes in every direction around the start.
- **A\*:** complete, and optimal because the Manhattan heuristic is admissible and consistent. In the measured results it returned paths with the same total cost as BFS and expanded fewer nodes, except when the destination was unreachable. An unreachable goal forces both algorithms to explore the whole reachable region, so the advantage disappears for that journey. In general, the advantage also becomes smaller when obstacles lie between the start and the goal. The per-journey comparison tables in Section 5 and the table in Section 6 give the exact numbers.
- **CSP:** backtracking search with MRV, the degree heuristic, LCV ordering and forward checking is complete, so it finds a feasible schedule whenever one exists and reports failure otherwise (for example, for a cycle of precedence constraints).
- **Limitations:** the cleaning order is fixed by the CSP and is not chosen to minimise the travel distance, so every journey is optimal but the whole tour need not be the shortest possible. All cells have the same cost. Only four-directional movement is supported.
- **Possible improvements:** treat the visiting order as an optimisation problem (add travel costs to the CSP, or compare all feasible orders), add arc consistency (AC-3) to the CSP, support cells with different costs, and use faster search methods such as bidirectional search for large grids.

## References
- Russell, S. and Norvig, P., *Artificial Intelligence: A Modern Approach*: graph search, A\* search and constraint satisfaction problems.
- CS-323 Artificial Intelligence course material and the Complex Engineering Problem description.